# **Selección del tracker base**

En este notebook, se muestra la comparativa realizada de los distintos trackers para seleccionar el tracker base.

Este documento tiene las siguientes partes:
- **Trackers comparados**: Tabla resumen de los trackers
- **Métricas utilizadas**: Explicación de las métricas utilizadas
- **Comparativa**: Comparativa de los trackers. Incluye el criterio de selección de los trackers, el criterio seguido en la comparativa entre trackers, la comparativa, y la elección del tracker base que se utilizará en el sistema.

## **Trackers comparados**

En la siguiente tabla se resumen las partes principales de cada algoritmo, de modo que se puedan comparar de forma rápida.

| **Tracker** | **Entrada** | **Predicción** | **Asociación detección-track** | **Otros aspectos** |
| :---: | :---: | :---: | :---: | :---: |
| **SORT** | *bounding boxes* de los objetos | Filtro de Kalman sobre posición y tamaño y velocidad del *bounding box* | Algoritmo húngaro sobre la matriz de costes del IoU | Existe un valor de IoU mínimo para considerar la asociación. Es rápido. Va mal ante oclusiones. |
| **ByteTrack** | *bounding boxes* de los objetos y sus confianzas | Filtro de Kalman, igual que SORT | Las detecciones de alta confianza se asocian igual que en SORT. Después, con los IDs que no se han asociado en la primera etapa, las detecciones de baja confianza se asocian usando también IoU. | Solo las detecciones de alta confianza no asociadas crean nuevos IDs. |
| **BoTSORT** | *bounding boxes* de los objetos y sus confianzas | Filtro de Kalman, igual que SORT. Las predicciones se ajustan según el movimiento estimado de cámara. | Las detecciones de alta confianza se asocian combinando IoU y la distancia de los embeddings de ReID. Las detecciones de baja confianza se asocian usando solo IoU. | Se usan umbrales en IoU y ReID para descartar asociaciones poco fiables. Solo las detecciones de alta confianza no asociadas, si superan un umbral, crean nuevos IDs. |
| **OC-SORT** | *bounding boxes* de los objetos | Filtro de Kalman con actualización adicional cuando un ID se vuelve a asociar tras varios frames sin asociarse. | Combina el IoU con el cambio de ángulo en la velocidad que sigue el ID en frames anteriores y la detección | Si se asocia un ID tras varios frames, se crea una trayectoria virtual de los frames perdidos que se usa en el filtro de Kalman. |
| **Deep OC-SORT** | *bounding boxes* de los objetos y sus confianzas | Calcula la predicción al igual que OC-SORT. Después, ajusta su posición según el movimiento estimado de cámara. | Suma con pesos del coste de IoU y el de apariencia (ReID). Cuanta menos confianza del detector, menos peso se le da al ReID. | Si las distancias de los embeddings entre las mejores asociaciones no son muy discriminativas, se reduce el peso del término de apariencia en el coste de asociación para las asociaciones candidatas que contienen esa detección o identidad. |
| **SAM2** | Vídeo, y puntos o *bounding boxes* que indiquen dónde están los objetos | Codificador de imágenes que codifica el frame, y un mecanismo de atención (frames anteriores condicionan al frame actual) | En vez de asociación como en métodos clásicos, se usa un decodificador de máscaras que usa la salida del mecanismo de atención para predecir las máscaras de los objetos | El paper indica que el modelo se puede confundir tras largas oclusiones, objetos muy cercanos, u objetos con apariencia muy similar. |

## **Métricas utilizadas**

En este apartado se indican las métricas seleccionadas para la comparación de algoritmos de seguimiento.

### **IDSW**

El número de intercambios de identidad (IDSW) corresponde al número de ocasiones en las que una trayectoria real que en frames anteriores estaba asociada a una identidad de predicción, pasa en el frame actual a asociarse a otra identidad de predicción, ya sea intercambiando identidad con otro identificador o creando un nuevo identificador de predicción.

Dado que el objetivo es obtener una detección robusta durante un partido completo, de 40 minutos de duración, el número de intercambios de identidad impacta muy negativamente en el objetivo.

### **IDs y GT\_IDs**

Dado que IDSW cuenta tanto creaciones de nuevas identidades como intercambio de identidades existentes, se contextualiza la métrica usando el número de identidades predichas (IDs), identidades reales (GT\_IDs), y su diferencia.

Esta métrica se relaciona directamente con el objetivo de evitar la fragmentación de las identidades de los jugadores.

### **IDTP, IDFN, y IDFP**

Para calcular el número de IDTP (*Identity True Positive*), IDFN (*Identity False Negative*), y IDFP (*Identity False Positive*), en primer lugar se deben asociar las trayectorias reales de los jugadores con las trayectorias predichas por el algoritmo de seguimiento. Esta asociación se debe realizar de forma que se asigne la mejor correspondencia entre trayectorias reales y predichas.

Por cada trayectoria real, se obtienen las métricas de la siguiente forma:
- **IDTP (Verdadero positivo)**: Número de frames en los que la trayectoria predicha asociada coincide con la real.
- **IDFN (Falso negativo)**: Número de frames en los que el objeto (trayectoria real) está visible en el frame, pero la trayectoria predicha no coincide con la real, ya sea porque se predice que el objeto no está presente o porque se predice que está en otro lugar.
- **IDFP (Falso positivo)**: Número de frames en los existe una trayectoria predicha que indica que un objeto se encuentra en el frame, pero esta no coincide con la trayectoria real asociada a esta, ya sea porque el objeto real no está presente en el frame o porque se predice que está en otro lugar.

Las métricas se calculan sumando los valores de IDTP, IDFN, y IDFP de cada trayectoria real, lo que permite obtener un valor global para cada métrica.

Estas métricas son clave para nuestro objetivo, descomponiendo los errores entre trayectorias reales no seguidas (IDFN), y trayectorias mal asociadas (IDFP). 

Cada intercambio de identidad entre dos identificadores existentes genera simultáneamente IDFN e IDFP. Sin embargo, si las trayectorias se vuelven a asociar correctamente, se produce un intercambio de identidad (IDSW, *Identity Switch*) que resulta positivo al aumentar el número de IDTP y dejar de incrementar el número de IDFN y IDFP.

### **IDF1**

IDF1 (*ID F1 score*) utiliza las métricas vistas anteriormente (*IDTP, IDFN, y IDFP*) de la siguiente forma:

$$\text{IDF}_1 = \frac{| |text{IDTP}|}{|\text{IDTP}|+0.5\times|\text{IDFP}| + 0.5\times|\text{IDFN}|}$$

A diferencia de métricas centradas en detección, IDF1 prioriza la consistencia de identidades a largo plazo, lo cual es clave para el seguimiento robusto de jugadores durante partidos completos.


### **AssA**

Se define de la siguiente forma:

$$\text{AssA}_\alpha = \frac{1}{|\text{TP}|} \sum_{c \in \{\text{TP}\}} A(c)$$

Donde TP representa el conjunto total de detecciones correctamente localizadas. $A(c)$ se define de la siguiente forma:

$$A(c) = \frac{|\text{TPA}(c)|}{|\text{TPA}(c)|+|\text{FNA}(c)|+|\text{FPA}(c)|}$$

Donde TPA($c$), FNA($c$) y FPA($c$) son las versiones específicas de IDTP, IDFN e IDFP para el emparejamiento entre trayectoria predicha y real $c$.

AssA cuantifica la precisión de la asociación de identidades a nivel de trayectorias individuales.

## **Comparativa**

### **Criterios de selección de trackers**

En esta sección, se analiza el comportamiento de los algoritmos de seguimiento BoT-SORT, ByteTrack, OC-SORT y Deep OC-SORT, junto con SAM2, en el contexto del baloncesto. 

Estos algoritmos se han seleccionado por distintos motivos:

- ByteTrack y OC-SORT representan métodos de seguimiento basados únicamente en el movimiento, un clásico en MOT (*Multiple Object Tracking*).
- BoT-SORT y Deep OC-SORT añaden la apariencia (ReID), lo que permite estudiar el impacto de la información visual en el seguimiento.
- SAM2 es un enfoque más moderno, basado en segmentación en lugar de detección de cajas donde se encuentran los jugadores, que suele conseguir mejores resultados a costa de un mayor coste computacional.

De este modo, la comparación cubre trackers clásicos basados en detección y movimiento, variantes que añaden apariencia, y un método reciente de segmentación, sin aumentar en exceso el número de algoritmos a analizar.

### **Criterios de comparativa entre trackers**


Durante la comparativa de trackers y la búsqueda de hiperparámetros, se utiliza IDF1 como métrica principal debido a que esta métrica prioriza la consistencia de identidades a largo plazo. Esta métrica tiene en cuenta tanto que se mantengan las trayectorias de los jugadores durante el mayor número de frames posible (IDTP), como que no aumente el número de IDFP e IDFN. El resto de métricas seleccionadas anteriormente se utilizan de forma complementaria con el fin de analizar el comportamiento más en detalle y evitar elecciones subóptimas derivadas de sólo considerar un valor.

Aunque el número de IDSW e IDFP son métricas clave para los objetivos principales de este trabajo, el tracker se utiliza después como base de un sistema más completo que añade la lectura del número de dorsal y la identificación del equipo de cada jugador para mejorar el seguimiento. Estas capas adicionales se añaden para corregir errores del tracker base. Por esto, si disminuyen los IDSW e IDFP en el tracker a costa de reducir el número de IDTP y aumentar el de IDFN, podría resultar contraproducente porque esto elimina la información que después se utiliza en el sistema para identificar que ha ocurrido un cambio de identidad y recuperar el identificador correcto. Es por esto que se utiliza IDF1 como métrica principal para seleccionar el tracker y sus hiperparámetros.

### **Comparativa entre trackers**

En la siguiente tabla se muestran las métricas de los trackers OC-SORT, Deep OC-SORT, ByteTrack, BoT-SORT y SAM2 en el conjunto de test. Además, se añade el número de frames por segundo (FPS) que tarda cada tracker de media porque el coste computacional es un factor importante.

| IDSW | IDs | GT\_IDs | IDTP | IDFN | IDFP | IDF1 | AssA | Modelo | FPS |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 540 | 535 | 150 | 69546 | 49279 | 42969 | 60.12 | 41.81 | OC-SORT | 122.91 |
| 432 | 408 | 150 | 71021 | 47804 | 39430 | 61.95 | 43.92 | Deep OC-SORT | 55.22 |
| 475 | 462 | 150 | 83754 | 35071 | 32515 | 71.25 | 51.99 | Botsort | 56.03 |
| 512 | 544 | 150 | 74190 | 44635 | 41861 | 63.17 | 40.91 | ByteTrack | 124.04 |
| 164 | 134 | 150 | 82403 | 36422 | 29460 | 71.44 | 70.12 | SAM2 | 4.66 |

Se observa que el número de FPS es aproximadamente el doble en ByteTrack y OC-SORT, que no utilizan información de apariencia, respecto a BoT-SORT y Deep OC-SORT, que sí la utilizan. Esto es coherente con el hecho de que el uso de un modelo de reidentificación aumenta el coste computacional a costa de mejorar el seguimiento.

Respecto a Deep OC-SORT y OC-SORT, que comparten la lógica basada en movimiento, el tracker Deep OC-SORT mejora las métricas pero no de forma muy elevada. Esto podría deberse a que el modelo de reidentificación, al tratar con jugadores con una apariencia muy similar (cada equipo tiene el mismo uniforme, y los uniformes sólo se diferencian en color), le es difícil distinguirlos, lo que limita el beneficio de la apariencia. Sin embargo, Deep OC-SORT obtiene mejores métricas que OC-SORT en todas las métricas, lo que confirma que la información de la apariencia es útil para mejorar el seguimiento, incluso en este contexto.

Por otra parte, ByteTrack y BoT-SORT obtienen un mayor IDF1 que OC-SORT y Deep OC-SORT. Una posible explicación es que estos trackers utilizan la confianza del modelo de detección de jugadores para dividir las detecciones en dos grupos (de alta confianza, y de baja confianza), gestionando de forma distinta cada uno, y debido a que se utiliza un modelo YOLO que proporciona esta confianza, ByteTrack y BoT-SORT se benefician de esta información. 

### **Selección del tracker base**

Las métricas de SAM2 son las mejores en el conjunto de test, con un IDF1 de 71.44, seguido de BoT-SORT con un IDF1 de 71.25. El resto de trackers, en comparativa con SAM2 y BoT-SORT, obtienen valores menores de IDF1, AssA, e IDTP, y mayores de IDSW, IDs, IDFN e IDFP, por lo que se descarta su uso como posibles trackers base del sistema.

El coste computacional de SAM2 es mucho mayor al de BoT-SORT, reduciendo la velocidad de 56 a 4.66 FPS. BoT-SORT genera un mayor número de IDSW e IDs que SAM2, mientras que el número de IDTP y de IDFN es relativamente similar. Aunque SAM2 obtiene mejores métricas globales (IDF1 y AssA), la diferencia de IDF1 no compensa el incremento de tiempo de computación para el caso de uso de este trabajo, en el que se busca un sistema capaz de procesar vídeos de 40 minutos en dispositivos informáticos más económicos que los utilizados a nivel profesional. 

Por las razones anteriores, se escoge BoT-SORT como tracker base para el sistema completo.